In [ ]:
# importing libraries
import numpy as np # for generating telemetry variables
import pandas as pd # for creating a data table
import matplotlib.pyplot as plt # for visualization

# Simulating a 30 minute race (30*60 = 1800 seconds)
time_seconds = np.arange(0, 1800)

# speed(km/h)
speed = (250-120 * np.abs(np.sin(time_seconds / 20)) + np.random.normal(0, 5, 1800))
# engine RPM
rpm = speed * 45 + np.random.normal(0, 100, 1800)
throttle = np.clip(np.random.normal(70, 15, 1800), 0, 100)
brake = np.clip(np.random.normal(10, 10, 1800), 0, 100)
# gear shifts(1-8)
gear = np.digitize(speed, bins=[50, 100, 150, 200, 250, 300, 350, 400])
steering_angle = np.random.normal(0, 10, 1800)

# **Telemetry data acquired for a driver in a qualifying race per lap**

In [ ]:
#telemtry variables in a dataframe
telemetry_data = pd.DataFrame({  "Time (s)": time_seconds,
                                  "Speed (km/h)": speed,
                                 "Engine RPM": rpm,
                                 "Throttle (%)": throttle,
                                 "Brake (%)": brake,
                                 "Gear": gear,
                                 "Steering Angle (deg)": steering_angle })
telemetry_data.head(10)

# **Visualization of Telemetry Data**

In [ ]:
# visualization of speed vs time per lap
plt.figure(figsize=(12,3))
plt.plot(telemetry_data["Time (s)"],telemetry_data["Speed (km/h)"],color='red', linewidth=2,linestyle='dashed',marker='.')
plt.title("Speed vs Time")
plt.xlabel("Time (s)")
plt.ylabel("Speed (km/h)")
plt.grid();
plt.show()


# steering angle vs time
# 0deg -> Straights
plt.figure(figsize=(9, 2))
plt.plot(telemetry_data["Time (s)"], telemetry_data["Steering Angle (deg)"],color='blue', linestyle='dashed',linewidth = 0.01,marker='.')
plt.title("Steering Angle vs Time")
plt.xlabel("Time (sec)")
plt.ylabel("Steering Angle (degrees)")
plt.grid();
plt.show()



# **Dry vs Wet track conditions**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dry track: faster, less braking
dry_speed = ( 240 - 110 * np.abs(np.sin(time_seconds / 22))  + np.random.normal(0, 4, 1800))

# Wet track: slower, deeper braking
wet_speed = (200 - 130 * np.abs(np.sin(time_seconds / 20))+ np.random.normal(0, 5, 1800))

# brake patterns (Dry = light, Wet = heavy)
dry_brake = (15 * np.abs(np.sin(time_seconds / 15))+ np.random.normal(0, 5, len(time_seconds)))
wet_brake = ( 35 * np.abs(np.sin(time_seconds / 15))+ np.random.normal(0, 5, len(time_seconds)))


In [ ]:
#Both dry and wet conditions in accordance with speed

conditions_data = pd.DataFrame({
                                "Time (s)": time_seconds,
                                "Dry Speed (km/h)": dry_speed,
                               "Wet Speed (km/h)": wet_speed,
                                "Dry Brake (%)": dry_brake,
                                "Wet Brake (%)": wet_brake,
                                })

conditions_data.head()


# **Visualiazation of Dry vs Wet Conditions**

In [ ]:
#Dry vs Wet Speed Comparison

plt.figure(figsize=(12,4))
plt.plot(time_seconds, dry_speed, label="Dry Speed", linestyle="-",color='purple',linewidth = '3')
plt.plot(time_seconds, wet_speed, label="Wet Speed", linestyle="-",color="blue",linewidth = '3')
plt.title("Dry vs Wet Speed Comparison")
plt.xlabel("Time (s)")
plt.ylabel("Speed (km/h")
plt.legend();
plt.grid();
plt.show()


In [ ]:
#Dry vs Wet brakes Comparison

plt.figure(figsize=(10, 4))
plt.plot(time_seconds, dry_brake, label='Dry Brake', color='red',linewidth = '2.5',)
plt.plot(time_seconds, wet_brake, label='Wet Brake', color='indigo',linewidth = '2.3',)
plt.title("Brake Pressure Comparison: Dry vs Wet Track")
plt.xlabel("Time (s)")
plt.ylabel("Brake Pressure (%)")
plt.legend()
plt.grid();
plt.show()


# **Comparitive Analysis using FastF1 Dataset**

In [ ]:
!pip install fastf1 --quiet
import os
os.makedirs("/content/cache", exist_ok=True)


In [ ]:
#importing fastf1 dataset and libraries
import fastf1
import pandas as pd
import matplotlib.pyplot as plt

fastf1.Cache.enable_cache('/content/cache')

#qualifying session for Verstappen (BAHRAIN GP) 2023
session = fastf1.get_session(2023, 'Bahrain', 'Q')
session.load()

#Qualifying wet session for Verstappen(JAPAN GP) 2022
session = fastf1.get_session(2022, 'Japan', 'Q')
session.load()

laps = session.laps.pick_driver('VER')

full_tel = pd.DataFrame()

for _, lap in laps.iterlaps():
    tel = lap.get_telemetry()
    tel["Time_s"] = tel["Time"].dt.total_seconds()
    full_tel = pd.concat([full_tel, tel], ignore_index=True)

full_tel = full_tel.sort_values("Time_s").reset_index(drop=True)
full_tel["Time_aligned"] = full_tel["Time_s"] - full_tel["Time_s"].iloc[0]

full_tel.head()

# **Predicted vs Actual F1 Speed**

In [ ]:
#predicted speed vs actual speed

plt.figure(figsize=(10,5))
#synthetically generated speed
plt.plot(telemetry_data["Time (s)"],telemetry_data["Speed (km/h)"],label="Predicted Speed",color='blue', linewidth='3',marker='.')

# actual qualifying speed from session
plt.plot(full_tel["Time_aligned"],full_tel["Speed"],label="Actual Speed",alpha=0.8,color = 'brown')

plt.title("Predicted vs Actual Speed")
plt.xlabel("Time (s)")
plt.ylabel("Speed (km/h)")
plt.legend()
plt.grid()
plt.show()


# **Predicted Wet Speed vs Actual Wet Speed**

In [ ]:
#loaded a wet session for wet track speed analysis
plt.figure(figsize=(10,5))

plt.plot(time_seconds, wet_speed,label="Predicted Wet Speed", linewidth=3,color = 'green')

plt.plot(full_tel["Time_aligned"], full_tel["Speed"],label="Actual Wet Speed",alpha=0.7,color = 'red')

plt.title("Predicted  Wet track Speed vs Actual Wet track Speed")
plt.xlabel("Time (s)")
plt.ylabel("Speed (km/h)")
plt.legend()
plt.grid()
plt.show()


# **Predicted vs Actual Wet track brake pressure**

In [ ]:
#predicted vs actual wet track brake pressure
plt.figure(figsize=(10,4))
full_tel["Brake_%"] = full_tel["Brake"] * 100

#loaded a wet session for wet track brake pressure analysis
plt.plot(time_seconds, wet_brake, label="Predicted Wet Brake", color='indigo',linewidth = '2.3',)

plt.plot(full_tel["Time_aligned"], full_tel["Brake_%"],label="Actual Wet Brake",alpha=0.7,color='black')

plt.title("Predicted vs Actual Wet track Brake pressure")
plt.xlabel("Time (s)")
plt.ylabel("Brake (%)")
plt.legend()
plt.grid()
plt.show()

# **Predicted vs Actual Dry track brake pressure**

In [ ]:
#predicted vs actual dry track brake pressure
full_tel["Brake_%"] = full_tel["Brake"] * 100
plt.figure(figsize=(10,4))

plt.plot(time_seconds, dry_brake,label="Predicted Dry Brake",color='red',linewidth = '2.5',)

plt.plot(full_tel["Time_aligned"], full_tel["Brake_%"], label="Actual Dry Brake", alpha=0.7,color = 'grey')

plt.title("predicted vs Actual Dry track Brake pressure")
plt.xlabel("Time (s)")
plt.ylabel("Brake (%)")
plt.legend()
plt.grid()
plt.show()


# **2026 Performance Equilization**

# 1. Power Unit(PU) Normalization

# 2. Aerodynamic Stabilization

In [ ]:
import numpy as np
import pandas as pd

def apply_2026_equalizer(df):
    eq_df = df.copy()

    # 1. PU Normalization: Detects 'Clipping' on straights
    # If Throttle > 95% and Speed > 250, we inject a manual override boost
    clipping_mask = (eq_df['Throttle (%)'] > 95) & (eq_df['Speed (km/h)'] > 250)
    eq_df.loc[clipping_mask, 'Speed (km/h)'] *= 1.08  # 8% Power Boost

    # 2. Aero Stabilization: Detects high-steering corners
# Based on steering_angle telematics, 15deg is stable for applying grip factor
    corner_mask = np.abs(eq_df['Steering Angle (deg)']) > 15
    eq_df.loc[corner_mask, 'Speed (km/h)'] *= 1.12 # 12% Grip Compensation

    return eq_df

# **Equalization Analysis with Old and New Conditions**

In [ ]:
# Applying both to the synthetic telemetry
equalized_synthetic = apply_2026_equalizer(telemetry_data)

# Applying to the Actual F1 data (full_tel)
# since FastF1 session is 2022/2023, this simulates Equalization with 2026 regs

# full_tel for the equalizer function
full_tel_prepared = full_tel.copy()
full_tel_prepared.rename(columns={'Throttle': 'Throttle (%)', 'Speed': 'Speed (km/h)'}, inplace=True)

#added a placeholder for 'Steering Angle (deg)'
if 'Steering Angle (deg)' not in full_tel_prepared.columns:
    full_tel_prepared['Steering Angle (deg)'] = 0

equalized_actual = apply_2026_equalizer(full_tel_prepared)

# **Visualization Between 2022/23 And 2026 Regulations**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Plot 1: The 'Regulated' 2026 EQ (Raw Data)
plt.plot(telemetry_data["Time (s)"][:250], telemetry_data["Speed (km/h)"][:250],
         label="2026 Regulated (Performance EQ)", color='red', alpha=0.5, linewidth = 3)

# Plot 2: Output of override boost and aero stabilization
plt.plot(equalized_synthetic["Time (s)"][:250], equalized_synthetic["Speed (km/h)"][:250],
         label="Equalized Battle (2022/23 Levels)", color='green', linewidth=2)

plt.title("Results: Performance Equalization for 2026 Regulations")
plt.xlabel("Time (s)")
plt.ylabel("Speed (km/h)")
plt.legend()
plt.grid(True)
plt.show()